In [ ]:
# Replicate the fune tuning the model change hyper parameters,
# Epoachs, Batch Sizes on 20.000 test data

In [ ]:
# imports

import os
import re
import json
from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI
from pricer.items  import Item
from pricer.evaluator import evaluate

In [ ]:
# environment

LITE_MODE = True

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

In [ ]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

In [ ]:
openai = OpenAI()


In [ ]:
fine_tune_train = train[:250]
fine_tune_validation = val[:50]

In [ ]:
def messages_for(item):
    message = f"""
    You are a product price estimation AI.
    Your task is to estimate the approximate market price in USD for a product based only on the provided description.
    Guidelines:
    - Use general global market knowledge to estimate the price.
    - Consider brand, materials, specifications, size, and category when mentioned.
    - If the description is vague, make a reasonable assumption based on common products in that category.
    - Do not explain your reasoning.
    - Do not include currency symbols or text.
    - Return only a single number representing the estimated price in USD.
    - Round to the nearest whole number.

    Output format:
    <number>

    Examples:

    Input: "Apple iPhone 14 Pro Max 256GB smartphone"
    Output:
    1100

    Input: "Wooden dining table for 6 people"
    Output:
    450

    Input: "Basic cotton t-shirt"
    Output:
    15
    """
    return [
        {"role": "user", "content": message},
        {"role": "assistant", "content": f"${item.price:.2f}"}
    ]

In [ ]:

def make_jsonl(items):
    result = ""
    for item in items:
        messages = messages_for(item)
        messages_str = json.dumps(messages)
        result += '{"messages": ' + messages_str +'}\n'
    return result.strip()

In [ ]:
# Convert the items into jsonl and write them to a file

def write_jsonl(items, filename):
    with open(filename, "w") as f:
        jsonl = make_jsonl(items)
        f.write(jsonl)

In [ ]:
write_jsonl(fine_tune_train, "jsonl/fine_tune_train.jsonl")


In [ ]:
write_jsonl(fine_tune_validation, "jsonl/fine_tune_validation.jsonl")


In [ ]:
with open("jsonl/fine_tune_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")

In [ ]:
with open("jsonl/fine_tune_validation.jsonl", "rb") as f:
    validation_file = openai.files.create(file=f, purpose="fine-tune")

In [ ]:
openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=validation_file.id,
    model="gpt-4.1-nano-2025-04-14",
    seed=42,
    hyperparameters={"n_epochs": 1, "batch_size": 1},
    # hyperparameters={"n_epochs": 1, "batch_size": 1}, Idea is to increase batch size but the sample number is only 250, so we can not increase batch size more than 1 and epochs too.
    suffix="pricer"
)

In [ ]:
openai.fine_tuning.jobs.list(limit=1)


In [ ]:
job_id = openai.fine_tuning.jobs.list(limit=1).data[0].id


In [ ]:
openai.fine_tuning.jobs.retrieve(job_id)


In [ ]:
openai.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10).data

In [ ]:
fine_tuned_model_name = openai.fine_tuning.jobs.retrieve(job_id).fine_tuned_model

In [ ]:
# The prompt

def test_messages_for(item):
    message = (f"Estimate the price of this product based on the description. Respond with the price in USD, no explanation\n\n "
               f"{item.summary}")
    return [
        {"role": "user", "content": message},
    ]

In [ ]:
test_messages_for(test[0])

In [ ]:
def gpt_4__1_nano_fine_tuned(item):
    response = openai.chat.completions.create(
        model=fine_tuned_model_name,
        messages=test_messages_for(item),
        max_tokens=7
    )
    return response.choices[0].message.content


print(test[0].price)
print(gpt_4__1_nano_fine_tuned(test[0]))
evaluate(gpt_4__1_nano_fine_tuned, test)